# BamiBERT hai giai đoạn theo từng công việc trên Kaggle

Giai đoạn một gán kiểu theo cặp biên. Giai đoạn hai tìm vai trò của đối số theo từng công việc. Lượt Kaggle dùng dataset riêng tư, chỉ nhận văn bản test để suy luận.

[Báo cáo tổng hợp](../reports/tong-hop-pipeline-smartnotice-v3.3-2026-11-23.md) · [Giao thức đánh giá](../../../docs/evaluation.md)

## 1. Cấu hình và dữ liệu

Nạp thông báo đã chuẩn bị và kiểm tra hash, nhóm, schema và các khoảng ký tự. Tham chiếu test v3.3 là chú giải lịch sử có máy hỗ trợ.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

repo = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tools/compare_notice_pipelines.py").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Mở notebook từ workspace SmartNotice.")
os.chdir(repo)
split = Path(os.environ.get("SMARTNOTICE_SPLIT_ROOT", "tmp/drive-run-20261123-fresh")).resolve()
base = Path(os.environ.get("SMARTNOTICE_BAMIBERT_DIR", "tmp/bamibert-checkpoint")).resolve()
destination = Path(os.environ.get(
    "SMARTNOTICE_NOTEBOOK_OUTPUT", "reports/generated/pipeline-notebooks-v33-20261123"
)).resolve()
baseline = Path(os.environ.get(
    "SMARTNOTICE_BASELINE", "reports/generated/bamibert/v3_3-2026-11-23-final-seed42"
)).resolve()


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def run_tool(script, *arguments, **environment):
    env = {**os.environ, "PYTHONHASHSEED": "42", "TOKENIZERS_PARALLELISM": "false", **environment}
    result = subprocess.run(
        [sys.executable, "-u", str(repo / "tools" / script), *map(str, arguments)],
        cwd=repo, env=env, capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    if result.returncode:
        print(result.stdout[-4000:], result.stderr[-4000:])
        result.check_returncode()
    print(result.stdout.strip())

In [2]:
from smartnotice.modeling.data import labeled_links, load_development, sha256, span_key

development, provenance = load_development(split)
for part, docs in development.items():
    print(part, {
        "notices": len(docs),
        "groups": len({d.group_id for d in docs}),
        "typed_spans": sum(len({span_key(s) for s in d.annotation.spans}) for d in docs),
        "links": sum(len(set(labeled_links(d.annotation))) for d in docs),
        "events": sum(len(d.annotation.events) for d in docs),
    })
print("Phân bổ:", provenance["split_counts"], "gold_ready:", provenance["gold_ready"])

train {'notices': 403, 'groups': 281, 'typed_spans': 4664, 'links': 3847, 'events': 1516}
validation {'notices': 93, 'groups': 77, 'typed_spans': 1078, 'links': 873, 'events': 357}
Phân bổ: {'train': 403, 'test': 126, 'validation': 93, 'domain_test': 62} gold_ready: False


## 2. Huấn luyện trên GPU và tạo dự đoán

[Mã huấn luyện](../../../tools/run_action_conditioned.py) chạy hai lần tinh chỉnh encoder, mỗi lần bốn epoch. Thiết lập các biến SMARTNOTICE_KAGGLE_KERNEL_DIR, SMARTNOTICE_KAGGLE_KERNEL và KAGGLE_CONFIG_DIR để gửi lượt mới với chế độ private. Đặt SMARTNOTICE_ACTION_RUN_DIR tới đầu ra GPU vừa tải để thực thi phần chấm cục bộ. Khi chạy cục bộ trên CUDA, có thể đặt SMARTNOTICE_ACTION_BUNDLE_DIR tới thư mục chứa experiment.zip.

In [3]:
out = destination / "run-08"
out.mkdir(parents=True, exist_ok=False)
bundle = os.environ.get("SMARTNOTICE_ACTION_BUNDLE_DIR")
kernel_dir = os.environ.get("SMARTNOTICE_KAGGLE_KERNEL_DIR")
remote_output = os.environ.get("SMARTNOTICE_ACTION_RUN_DIR")
if remote_output:
    import shutil

    for name in ("run.json", "span-history.json", "role-history.json", "validation-predictions.json",
                 "test-predictions.json"):
        shutil.copyfile(Path(remote_output) / name, out / name)
    print("Dùng dự đoán của lượt Kaggle đã hoàn tất:", Path(remote_output).name)
elif kernel_dir:
    import time

    kernel = os.environ["SMARTNOTICE_KAGGLE_KERNEL"]
    metadata = read_json(Path(kernel_dir) / "kernel-metadata.json")
    assert metadata["id"] == kernel and metadata["is_private"] is True
    assert metadata["enable_gpu"] is True and metadata["enable_internet"] is False
    assert sha256(Path(kernel_dir) / metadata["code_file"]) == sha256(repo / "tools/run_action_conditioned.py")
    subprocess.run(["kaggle", "kernels", "push", "-p", kernel_dir], check=True)
    for _ in range(180):
        response = subprocess.check_output(["kaggle", "kernels", "status", kernel], text=True)
        if "KernelWorkerStatus.COMPLETE" in response:
            break
        if any(f"KernelWorkerStatus.{state}" in response for state in ("ERROR", "CANCELLED")):
            raise RuntimeError(response)
        time.sleep(30)
    else:
        raise TimeoutError("Lượt Kaggle chưa hoàn tất.")
    subprocess.run(["kaggle", "kernels", "output", kernel, "-p", str(out)], check=True)
elif bundle:
    run_tool("run_action_conditioned.py", "--data", bundle, "--output", out)
else:
    raise RuntimeError("Cấu hình lượt Kaggle, đầu ra GPU đã tải hoặc gói riêng tư chạy CUDA cục bộ.")

Dùng dự đoán của lượt Kaggle đã hoàn tất: kaggle-action-v33-20261123


In [4]:
run = read_json(out / "run.json")
assert run["provenance"]["hashes"] == provenance["hashes"]
print("GPU:", run["gpu"], "seed:", run["seed"], "epoch mỗi giai đoạn:", run["epochs_each"])
for stage in ("span", "role"):
    print(stage, [round(e["validation"]["f1"], 4) for e in read_json(out / f"{stage}-history.json")])
print("F1 khi cấp ACTION chú giải:", run["oracle_validation"]["f1"])

GPU: Tesla T4 seed: 42 epoch mỗi giai đoạn: 4
span [0.3681, 0.4016, 0.4371, 0.4418]
role [0.1914, 0.2088, 0.2404, 0.2513]
F1 khi cấp ACTION chú giải: 0.4621260343730108


## 3. Chấm cục bộ trên validation và test thường

Trình chấm mới mở chú giải test ở bước này, kiểm tra hash và tính điểm từ dự đoán offset.

In [5]:
from smartnotice.modeling.experiment import write_summary
from smartnotice.modeling.metrics import prf


def show(scores):
    span = scores["span"]["micro"]
    link = scores["link"]["micro"]
    event = scores["event"]
    for caption, score in (("Cụm có kiểu", span), ("Quan hệ", link), ("Sự kiện", event)):
        print(
            f"{caption}: F1 {score['f1']:.2%} | P {score['precision']:.2%} | "
            f"R {score['recall']:.2%} | đúng/dự đoán/tham chiếu "
            f"{score['matched']}/{score['predicted']}/{score['expected']}"
        )
    print("Ngày tường minh:", scores["dates"]["explicit"])


def without_reference(docs):
    from smartnotice.modeling.data import Document
    from smartnotice.schemas import Annotation

    return [
        Document(d.notice, Annotation(
            doc_id=d.notice.doc_id, annotator_id="inference", spans=[], events=[]
        ), d.group_id)
        for d in docs
    ]


def boundary_diagnostics(docs, predictions):
    expected, predicted = {}, {}
    for d in docs:
        doc_id = d.notice.doc_id
        for target, spans in (
            (expected, map(span_key, d.annotation.spans)), (predicted, predictions[doc_id][0])
        ):
            for begin, end, kind in spans:
                target.setdefault((doc_id, begin, end), set()).add(kind)
    matched = expected.keys() & predicted.keys()
    return {
        "untyped": prf(set(expected), set(predicted)),
        "correct_type_sets": sum(expected[key] == predicted[key] for key in matched),
        "matched_boundaries": len(matched),
    }

In [6]:
run_tool("score_action_conditioned.py", "--split", split, "--run", out)
scores = read_json(out / "local_scores.json")
for part in ("validation", "test"):
    print(part)
    show(scores[part])
    print("Ranh giới không tính kiểu:", scores[part]["untyped_boundary"])
    print("Tập kiểu đúng trên ranh giới khớp:", scores[part]["correct_label_sets_on_matching_boundaries"])

validation {'precision': 0.24858115777525538, 'recall': 0.2508591065292096, 'f1': 0.24971493728620298, 'matched': 219, 'predicted': 881, 'expected': 873} {'precision': 0.072992700729927, 'recall': 0.08403361344537816, 'f1': 0.078125, 'matched': 30, 'predicted': 411, 'expected': 357}
test {'precision': 0.23794871794871794, 'recall': 0.20191470844212359, 'f1': 0.2184557438794727, 'matched': 232, 'predicted': 975, 'expected': 1149} {'precision': 0.029880478087649404, 'recall': 0.03205128205128205, 'f1': 0.030927835051546393, 'matched': 15, 'predicted': 502, 'expected': 468}
validation
Cụm có kiểu: F1 44.34% | P 39.49% | R 50.56% | đúng/dự đoán/tham chiếu 545/1380/1078
Quan hệ: F1 24.97% | P 24.86% | R 25.09% | đúng/dự đoán/tham chiếu 219/881/873
Sự kiện: F1 7.81% | P 7.30% | R 8.40% | đúng/dự đoán/tham chiếu 30/411/357
Ngày tường minh: {'expected_links': 135, 'matched_links': 28, 'correct_dates': 11, 'matched_date_accuracy': 0.39285714285714285, 'matched_inferred_flag_accuracy': 1.0, 'end